## 6 Итоговая оценка

**Входы**

- Выбранная модель, порог и test

**Выходы**

- Метрики, графики и разбор ошибок

**Что делаем**

Проверим готовое решение на test Модель и порог на этом шаге уже не меняем

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Как читать результаты

Используем те же TP, FP, FN и TN, что определили в постановке задачи Дефолт — положительный исход, поэтому FN означает пропущенный дефолт, а FP — ложную тревогу

| Метрика | Формула | Что показывает |
| --- | --- | --- |
| Accuracy | $\frac{TP+TN}{TP+FP+FN+TN}$ | Доля правильных ответов по всем кредитам |
| Precision | $\frac{TP}{TP+FP}$ | Доля настоящих дефолтов среди отмеченных моделью |
| Recall | $\frac{TP}{TP+FN}$ | Доля найденных дефолтов среди всех настоящих |
| F1 | $\frac{2TP}{2TP+FP+FN}$ | Объединяет precision и recall, учитывая их на равных |
| F2 | $\frac{5TP}{5TP+FP+4FN}$ | Объединяет precision и recall с большим весом recall |
| Balanced accuracy | $\frac{1}{2}\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right)$ | Средняя полнота для двух классов: многочисленный класс не перевешивает редкий |

Если знаменатель precision, recall, F1 или F2 равен нулю, в расчётах принимаем значение такой метрики равным 0

Матрица ошибок показывает все четыре количества В строках стоит фактический исход, в столбцах — прогноз: TN и FP в первой строке, FN и TP во второй

Ещё используем метрики, которые оценивают прогнозы риска при разных порогах или без перевода в класс

| Метрика | Что показывает |
| --- | --- |
| AP | Обобщает precision при изменении recall по разным порогам; больше — лучше |
| ROC-AUC | Как часто кредит с дефолтом получает больший риск, чем кредит без дефолта, при попарном сравнении 0,5 — случайный порядок, 1 — идеальный |
| Brier | Средний квадрат разницы между прогнозом риска и фактическим исходом 0/1; меньше — лучше |

ROC-кривая сопоставляет полноту $\frac{TP}{TP+FN}$ с долей ложных тревог среди кредитов без дефолта $\frac{FP}{FP+TN}$ PR-кривая — точность $\frac{TP}{TP+FP}$ с полнотой $\frac{TP}{TP+FN}$ при разных порогах

**Калибровка** показывает, согласуются ли оценки риска с частотой событий Например, среди заявок с риском около 20% хотелось бы видеть примерно 20% дефолтов График ниже только проверяет это: по test ничего не настраиваем

Чтобы понять разброс метрик, используем **bootstrap**: 300 раз собираем тестовый набор случайным выбором строк с повторами Число кредитов каждого класса сохраняем; модель и порог не меняем Показываем границы, между которыми лежат средние 95% полученных значений

Это ориентир для этой модели и этого test Он не учитывает новое обучение, возможные повторы клиентов или изменение данных со временем

In [ ]:
# dev-only: загрузчик Python-кода; в сборке заменяется содержимым файла
exec(compile(__import__('pathlib').Path('src/notebooks/_code/evaluate.py').read_text(encoding='utf-8'), 'src/notebooks/_code/evaluate.py', 'exec'), globals())

#### Что ограничивает выводы

Мы проверили модель на одном случайном разбиении Kaggle-датасета Не знаем время заявок, период наблюдения дефолта и то, встречается ли один человек несколько раз Также неизвестно, насколько эти данные похожи на заявки конкретного банка Редкие грейды представлены небольшим числом записей

Грейд и ставка должны быть известны до прогноза Если нужна оценка до их назначения, состав признаков придётся пересмотреть и провести новый эксперимент

Для практического применения нужны проверки на более поздних и внешних данных, оценка стоимости ошибок и качества для разных групп заёмщиков Полученные метрики сами по себе не обосновывают автоматический отказ в кредите